In [14]:
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer
from datasets import load_dataset

In [15]:
torch.manual_seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 64
block_size = 256 # max context length
n_embd = 384
lr = 3e-4
eval_iters = 200
eval_interval = 500
max_iters = 5000
dropout = 0.2
n_dlayers = 6
n_elayers = 6
n_heads = 6
vocab_size = 16384
n_blocks = 6

In [16]:
tokenizer = Tokenizer.from_file("data/tokenizer-wiki.json")

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [17]:
def encode_split(split):
    dataset = ds[split]
    token_chunks = []

    for start in range(0, len(dataset), batch_size):
        texts = dataset[start : start + batch_size]["text"]
        encodings = tokenizer.encode_batch(texts)
        token_ids = [token_id for encoding in encodings for token_id in encoding.ids]
        if token_ids:
            token_chunks.append(torch.tensor(token_ids, dtype=torch.int32))

    if not token_chunks:
        raise ValueError(f"Dataset split '{split}' contains no tokens")

    return torch.cat(token_chunks)

dtr = encode_split("train")
dval = encode_split("validation")

In [18]:
def get_batch(split="train"):
    data = dtr if split == "train" else dval

    starts = torch.randint(data.numel() - block_size, (batch_size,))
    offsets = torch.arange(block_size)
    positions = starts[:, None] + offsets[None, :]

    x = data[positions].to(device=device, dtype=torch.long)
    y = data[positions + 1].to(device=device, dtype=torch.long)

    return x, y

In [19]:
get_batch()

(tensor([[ 5162,  7740, 12025,  ...,    64, 13707, 16237],
         [15373,  6982,  8193,  ..., 10566,  5068, 10388],
         [11695,    64, 16116,  ...,  6967, 14069,    35],
         ...,
         [ 8434,  5026,  7282,  ...,  5153,    64,    18],
         [ 9684,  8932,     7,  ...,     7, 12706,  7213],
         [   28,    28,    28,  ..., 14316,  6686,  5641]]),
 tensor([[ 7740, 12025,  5285,  ..., 13707, 16237,  5430],
         [ 6982,  8193,  5343,  ...,  5068, 10388,    13],
         [   64, 16116,     7,  ..., 14069,    35,  5420],
         ...,
         [ 5026,  7282,  8947,  ...,    64,    18,  1687],
         [ 8932,     7,  6827,  ..., 12706,  7213,     8],
         [   28,    28,    28,  ...,  6686,  5641,    11]]))

In [20]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, n_embd)
        # Add learnable positional embeddings
        self.pos_embd = nn.Embedding(block_size, n_embd)

        # Initialize MultiheadAttention with batch_first=True
        self.mha1 = nn.MultiheadAttention(n_embd, n_heads, dropout, batch_first=True)
        self.mha2 = nn.MultiheadAttention(n_embd, n_heads, dropout, batch_first=True)

        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device_idx = idx.device

        tok_emb = self.embd(idx.long())
        pos_emb = self.pos_embd(torch.arange(T, device=device_idx))
        x = tok_emb + pos_emb
        
        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=device_idx), diagonal=1
        )

        # mha1 takes (batch, seq_len, emb_dim)
        attn_out, weights = self.mha1(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)

        # mha2 takes (batch, seq_len, emb_dim)
        attn_out, weights = self.mha2(x, x, x, attn_mask=causal_mask)
        x = self.ln2(x + attn_out)

        x = self.ln3(x + self.ffwd(x))

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [21]:
model = Transformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [22]:
context = tokenizer.encode("Hello world").ids
# logits, loss = model(torch.tensor(context, dtype=torch.long, device=device))

output = model.generate(torch.tensor(context, dtype=torch.long, device=device).unsqueeze(0), max_new_tokens=100)
tokenizer.decode(output[0].tolist())

'Hel lo world É ways prepar ᡵ gam brilli ill collaborated ict ament ᾰ і application convoy rang Fer 週 evolved 謎 gets hall Nev series 磐 unknown Cath cept Cameron ว models ण Born 式 helping 関 Enter 免 ous raid diamond February mitted 毬 夜 私 hearing Museum dies straight powerful minute motif 杏 advantage 식 Start Formation ە 俊 楼 sher being prot 39 蝕 counties approval 姊 ラ aters sculpture ons particles 渦 cruc exter thetic Last 能 due 止 EC ẃ orton extending တ doct 戸 Gal amb igin Author gameplay Virginia Taylor azines bone aria 惑 Le'

In [43]:
get_batch()

TypeError: expected Tensor as element 0 in argument 0, but got list

In [23]:
for step in range(max_iters):
    Xb, yb = get_batch()

    logits, loss = model(Xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f'{step} / {max_iters}: {loss.item()}')

print("Final loss:", loss)

KeyboardInterrupt: 